In [11]:
import pandas as pd
import numpy as np
import os
from torchvision import datasets, transforms

# Trees Disease Image Classification with AlexNet

### Link Video: https://drive.google.com/drive/folders/1Ma6UjX-uzKEImL1mx47KlfJcBvn8zQel?usp=sharing

## Buka Folder dan read semua gambar yang ada di folder data

In [15]:
dt = "data"
tf = transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor()])

dataset = datasets.ImageFolder(dt,tf)

## Check isi dataset

In [18]:
print(dataset.classes)
print(len(dataset.classes))
dataset

['Cherry Leaf Scorch', 'Cherry Normal leaf', 'Cherry brown_spot', 'Cherry purple leaf spot', 'Cherry_shot hole disease']
5


Dataset ImageFolder
    Number of datapoints: 3642
    Root location: data
    StandardTransform
Transform: Compose(
               Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
           )

## Check jumlah folder tiap jenis (sudah di cek di folder asli dan benar)

In [112]:
classct = {}

for i in dataset.classes:
    path = os.path.join(dt,i)
    classct[i] = len(os.listdir(path))

classct

{'Cherry Leaf Scorch': 1101,
 'Cherry Normal leaf': 500,
 'Cherry brown_spot': 614,
 'Cherry purple leaf spot': 987,
 'Cherry_shot hole disease': 440}

## Explore resolusi seperti width dan height dan ratio dari keduanya

In [114]:
from PIL import Image

width = []
height = []

for i in dataset.classes:
    path = os.path.join(dt,i)
    image = os.listdir(path)
    for j in image:
        pathimg = os.path.join(path,j)
        img = Image.open(pathimg)
        w,h = img.size
        width.append(w)
        height.append(h)

print(np.mean(width))
print(np.mean(height))

800.0
1000.0


In [115]:
ratio = (np.mean(width)/(np.mean(height)))
print(ratio)

0.8


## Split jadi Train, test dan validate

In [116]:
from torch.utils.data import DataLoader, random_split

tot = len(dataset)

trainsize = int(0.7 * tot)
vsize = int(0.15 * tot)
testsize = tot - trainsize-vsize

train,val,test = random_split(dataset,[trainsize,vsize,testsize])


## Load data train, validate, test

In [117]:
trainload = DataLoader(
    train,
    batch_size=32,
    shuffle=True
)

vload = DataLoader(
    val,
    batch_size=32,
    shuffle=False
)

testload = DataLoader(
    test,
    batch_size=32,
    shuffle=False
)

## Model pertama manual

In [118]:
import torch
import torch.nn as nn

class AlexNet(nn.Module):
    def __init__(self, nc=5):
        super(AlexNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,96,kernel_size=11,stride=4,padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3,stride=2),

            nn.Conv2d(96,256,kernel_size=5,padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3,stride=2),

            nn.Conv2d(256,384,kernel_size=3,padding=1),
            nn.ReLU(),

            nn.Conv2d(384,384,kernel_size=3,padding=1),
            nn.ReLU(),

            nn.Conv2d(384,256,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3,stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(256*6*6, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096,4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096,nc)
        )
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)

        return x

## Jalankan Model, define learning rate, epoch dan loss function

In [119]:
nc = len(dataset.classes)
model = AlexNet(nc)

In [120]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

epochs = 10

for e in range(epochs):
    model.train()
    rloss = 0

    for i, j in trainload:
        optimizer.zero_grad()
        outputs = model(i)
        loss = criterion(
            outputs,
            j
        )
        loss.backward()
        optimizer.step()
        rloss += loss.item()

    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for i, j in vload:
            outputs = model(i)
            _, preds = torch.max(outputs,1)
            total += j.size(0)
            correct += (preds == j).sum().item()        
        vacc = correct / total
    
    print(
        f"Epoch {e+1}/{epochs}, "
        f"Loss: {rloss/len(trainload):.4f},"
        f"Val Acc: {vacc:.4f}"
)

Epoch 1/10, Loss: 1.4627,Val Acc: 0.4322
Epoch 2/10, Loss: 1.0031,Val Acc: 0.6465
Epoch 3/10, Loss: 0.7408,Val Acc: 0.7326
Epoch 4/10, Loss: 0.5698,Val Acc: 0.7875
Epoch 5/10, Loss: 0.5095,Val Acc: 0.8150
Epoch 6/10, Loss: 0.4463,Val Acc: 0.8407
Epoch 7/10, Loss: 0.3343,Val Acc: 0.8352
Epoch 8/10, Loss: 0.3063,Val Acc: 0.8938
Epoch 9/10, Loss: 0.2619,Val Acc: 0.8828
Epoch 10/10, Loss: 0.2196,Val Acc: 0.9066


## Evaluasi Model1

In [128]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score

pred1 = []
label1 = []

model.eval()

with torch.no_grad():
    for i, j in testload:
        outputs = model(i)
        _, preds = torch.max(outputs,1)

        pred1.extend(preds.cpu().numpy())
        label1.extend(j.cpu().numpy())

acc = accuracy_score(label1,pred1)

precision = precision_score(label1,pred1,average='weighted')

recall = recall_score(label1,pred1,average='weighted')

print(acc)
print(precision)
print(recall)


0.9140767824497258
0.9178323247544159
0.9140767824497258


## Confusion Matrix

In [129]:
import pandas as pd

cm = confusion_matrix(label1,pred1)

cname = dataset.classes

conf=pd.DataFrame(cm,index=cname,columns=cname)

conf

,Cherry Leaf Scorch,Cherry Normal leaf,Cherry brown_spot,Cherry purple leaf spot,Cherry_shot hole disease
Cherry Leaf Scorch,169,2,1,0,0
Cherry Normal leaf,2,70,2,0,6
Cherry brown_spot,6,11,71,5,0
Cherry purple leaf spot,2,0,1,139,0
Cherry_shot hole disease,0,9,0,0,51


## Classification report

In [130]:
print(classification_report(label1,pred1,target_names=cname))

                          precision    recall  f1-score   support

      Cherry Leaf Scorch       0.94      0.98      0.96       172
      Cherry Normal leaf       0.76      0.88      0.81        80
       Cherry brown_spot       0.95      0.76      0.85        93
 Cherry purple leaf spot       0.97      0.98      0.97       142
Cherry_shot hole disease       0.89      0.85      0.87        60

                accuracy                           0.91       547
               macro avg       0.90      0.89      0.89       547
            weighted avg       0.92      0.91      0.91       547



## Model 2, tambahkan batchnorm dan kecilkan output channel, ubah alur di classifier

In [152]:
class AlexNet2(nn.Module):
    def __init__(self, nc=5):
        super(AlexNet2, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,64,kernel_size=11,stride=4,padding=2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3,stride=2),

            nn.Conv2d(64,192,kernel_size=5,padding=2),
            nn.BatchNorm2d(192),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3,stride=2),

            nn.Conv2d(192,384,kernel_size=3,padding=1),
            nn.BatchNorm2d(384),
            nn.ReLU(),

            nn.Conv2d(384,256,kernel_size=3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.Conv2d(256,256,kernel_size=3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3,stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 6 * 6, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, nc)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)

        return x


## Jalankan model 2, define learning rate(diubah jadi 0.05) dan loss function(sama) dan epoch jadi 15

In [142]:
nc = len(dataset.classes)
model2 = AlexNet2(nc)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model2.parameters(),
    lr=0.0005
)

epochs = 15

for e in range(epochs):
    model2.train()
    rloss = 0

    for i, j in trainload:
        optimizer.zero_grad()
        outputs = model2(i)
        loss = criterion(outputs,j)
        
        loss.backward()
        optimizer.step()
        rloss += loss.item()

    model2.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for i, j in vload:
            outputs = model2(i)
            _, preds = torch.max(outputs,1)
            total += j.size(0)
            correct += (preds == j).sum().item()
    
    vacc = correct / total

    print(
        f"Epoch {e+1}/{epochs}, "
        f"Loss: {rloss/len(trainload):.4f},"
        f"Val Acc: {vacc:.4f}"
    )

Epoch 1/15, Loss: 1.0847,Val Acc: 0.3040
Epoch 2/15, Loss: 0.5455,Val Acc: 0.7363
Epoch 3/15, Loss: 0.4145,Val Acc: 0.8223
Epoch 4/15, Loss: 0.3598,Val Acc: 0.8608
Epoch 5/15, Loss: 0.3619,Val Acc: 0.7766
Epoch 6/15, Loss: 0.2929,Val Acc: 0.8663
Epoch 7/15, Loss: 0.2572,Val Acc: 0.8956
Epoch 8/15, Loss: 0.2396,Val Acc: 0.8516
Epoch 9/15, Loss: 0.2157,Val Acc: 0.9011
Epoch 10/15, Loss: 0.2434,Val Acc: 0.8773
Epoch 11/15, Loss: 0.2169,Val Acc: 0.9451
Epoch 12/15, Loss: 0.1776,Val Acc: 0.8718
Epoch 13/15, Loss: 0.1488,Val Acc: 0.9267
Epoch 14/15, Loss: 0.1722,Val Acc: 0.8993
Epoch 15/15, Loss: 0.1532,Val Acc: 0.8700


## Evaluasi test

In [143]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score

pred2 = []
label2 = []

model2.eval()

with torch.no_grad():
    for i, j in testload:
        outputs = model2(i)
        _, preds = torch.max(outputs,1)

        pred2.extend(preds.cpu().numpy())
        label2.extend(j.cpu().numpy())

acc2 = accuracy_score(label2,pred2)

precision2 = precision_score(label2,pred2,average='weighted')

recall2 = recall_score(label2,pred2,average='weighted')

print(acc2)
print(precision2)
print(recall2)

0.850091407678245
0.868454181282523
0.850091407678245


## Confusion Matrix

In [144]:
import pandas as pd

cm2 = confusion_matrix(label2,pred2)

cname = dataset.classes

conf2=pd.DataFrame(cm2,index=cname,columns=cname)

conf2

,Cherry Leaf Scorch,Cherry Normal leaf,Cherry brown_spot,Cherry purple leaf spot,Cherry_shot hole disease
Cherry Leaf Scorch,156,0,1,15,0
Cherry Normal leaf,1,48,9,11,11
Cherry brown_spot,2,2,68,21,0
Cherry purple leaf spot,0,0,0,142,0
Cherry_shot hole disease,0,3,0,6,51


## Classification Report

In [145]:
print(classification_report(label2,pred2,target_names=cname))

                          precision    recall  f1-score   support

      Cherry Leaf Scorch       0.98      0.91      0.94       172
      Cherry Normal leaf       0.91      0.60      0.72        80
       Cherry brown_spot       0.87      0.73      0.80        93
 Cherry purple leaf spot       0.73      1.00      0.84       142
Cherry_shot hole disease       0.82      0.85      0.84        60

                accuracy                           0.85       547
               macro avg       0.86      0.82      0.83       547
            weighted avg       0.87      0.85      0.85       547



## Kesimpulan, model pertama memiliki performa lebih baik meskipun memiliki epoch yang lebih kecil dan learning rate lebih kecil